In [6]:
import pandas as pd
import numpy as np
import re

# ==========================
# LOAD DATA
# ==========================

file="../5.Near Miss for 2015.xlsx"
sheet="Near Miss"

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

print("Original Shape:", df.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

df=df.replace(
    [
        "NA",
        "N/A",
        "na",
        "n/a",
        "NULL",
        "null",
        "nan",
        "Not Mentioned",
        "not mentioned",
        "NOT MENTIONED"
    ],
    np.nan
)

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    if (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    ):
        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

# ==========================
# REMOVE ROWS ONLY IF BOTH
# DESCRIPTION COLUMNS EMPTY
# ==========================

detail_col=None
event_col=None

for col in df.columns:

    name=col.lower()

    if "details_of_near_miss" in name:
        detail_col=col

    elif (
        "description" in name
        and
        "event" in name
    ):
        event_col=col

if detail_col and event_col:

    before=len(df)

    keep_mask=(
        (
            df[detail_col]
            .fillna("")
            .str.strip()
            !=""
        )
        |
        (
            df[event_col]
            .fillna("")
            .str.strip()
            !=""
        )
    )

    df=df[keep_mask]

    print(
        "\nRows Removed (Both description fields empty):",
        before-len(df)
    )

elif detail_col:

    before=len(df)

    df=df[
        df[detail_col]
        .fillna("")
        .str.strip()
        !=""
    ]

    print(
        "\nRows Removed (Empty Details of near miss):",
        before-len(df)
    )

elif event_col:

    before=len(df)

    df=df[
        df[event_col]
        .fillna("")
        .str.strip()
        !=""
    ]

    print(
        "\nRows Removed (Empty Description):",
        before-len(df)
    )

# ==========================
# FORMAT DATE COLUMNS
# ==========================

date_cols=[]

for col in df.columns:

    clean=col.lower()

    if (
        "date" in clean
        or
        "time_of_occurrence" in clean
        or
        "date_&_time" in clean
    ):
        date_cols.append(col)

def clean_date(v):

    if pd.isna(v):
        return np.nan

    v=str(v).strip()

    if v.lower() in [
        "na",
        "n/a",
        "not mentioned",
        "nan"
    ]:
        return np.nan

    # already correct → keep
    if re.match(
        r"^\d{2}-[A-Za-z]{3}-\d{2}$",
        v
    ):
        return v

    try:

        d=pd.to_datetime(
            v,
            errors="coerce"
        )

        if pd.notna(d):

            return d.strftime(
                "%d-%b-%y"
            )

    except:
        pass

    return np.nan

for col in date_cols:

    df[col]=df[col].apply(clean_date)

print("\nFormatted Date Columns:")
print(date_cols)

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print("\nDuplicates Removed:",duplicates)

# ==========================
# RESET SERIAL NUMBER
# ==========================

si_col=None

for col in df.columns:

    c=(
        col.lower()
        .replace("_","")
        .replace(":","")
    )

    if c in [
        "slno",
        "sino",
        "serialno"
    ]:
        si_col=col
        break

df=df.reset_index(
    drop=True
)

if si_col:

    df[si_col]=range(
        1,
        len(df)+1
    )

    print("\nSI_No reordered")

# ==========================
# FINAL MISSING SUMMARY
# ==========================

missing=(
    df
    .replace("",np.nan)
    .isna()
    .sum()
)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nFinal Columns:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_5_Near_Miss.xlsx"

df.to_excel(
    output,
    index=False
)

print("\nSaved:",output)

Original Shape: (630, 32)

Removed Empty Columns:
[]

Rows Removed (Both description fields empty): 0

Formatted Date Columns:
['Ref_num/Report_date', 'Date_&_Time_of_occurrence']

Duplicates Removed: 0

SI_No reordered

Missing Values Summary:
Ref_num/Report_date                             630
Port_Name                                       423
Details_of_near_miss                             35
Description_of_event_leading_to_the_incident    150
Immediate_action_taken                           27
Potential_extent_of_damage/injury               229
Cause_Analysis                                   19
Proposed_Corrective_Action                       97
Details_of_potential_loss_Category               59
Immediate_action_initiated                       22
Learning_Potential                               52
Severity_of_incident                             52
Corrective_Action(s)                            519
dtype: int64

Final Shape:
(630, 32)

Final Columns:
['Sl_No', 'Ref_num/Report_